# MyAI K5 — Data 100M: stream + 32k BPE + pack-512 (CPU-only)
10x scale-up for base strengthening: ~100M-token FineWeb sample, 32k indexed BPE, pack-512 + shards. Outputs feed K6 (resume-100M). ~1.5-2h, no GPU.

In [ ]:
!git clone https://github.com/aloksingh9667/llm.git myai 2>&1 | tail -1
%cd /kaggle/working/myai
!pip install -q -r requirements/train.txt 2>&1 | tail -1
!python scripts/stream_fineweb.py --max-tokens 100000000 --out data/raw/fineweb-100M.txt
!python scripts/prepare_tokenizer_corpus.py

In [ ]:
!python scripts/train_tokenizer.py --corpus data/tokenizer_corpus/train.txt --vocab-size 32000 --out data/tokenizer_corpus/tokenizer-32k.json --fast
!test -f data/tokenizer_corpus/tokenizer-32k.json && echo TOKENIZER_OK

In [ ]:
%%writefile configs/data-pack-fw100m.yaml
version: "1.0.0"
corpus_path: data/tokenizer_corpus/train.txt
tokenizer_path: data/tokenizer_corpus/tokenizer-32k.json
output_dir: data/processed/fw100m-32k
seq_len: 512
val_ratio: 0.01

In [ ]:
!python scripts/prepare_sequences.py --config configs/data-pack-fw100m.yaml --rows-per-shard 2048
!test -f data/processed/fw100m-32k/manifest.json && echo PACK_OK

In [ ]:
!cp data/tokenizer_corpus/tokenizer-32k.json /kaggle/working/
!cp data/processed/fw100m-32k/manifest.json /kaggle/working/pack100m-manifest.json
!cp data/processed/fw100m-32k/index.json /kaggle/working/pack100m-index.json
!python -c "import json; m=json.load(open('/kaggle/working/pack100m-manifest.json')); print('tokens:', m['total_tokens'], 'train:', m['train_sequences'], 'val:', m['val_sequences'])"